# 📊 Bloque 2 — Sesión 3: Visualización y perfilado automático

**Módulo:** Depuración, limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos  
**Duración estimada:** ~3 horas.

---

### Contenidos de esta sesión
- **2.6** Herramientas de visualización: matplotlib y seaborn
- **2.7** Perfilado automático de datos: ydata-profiling

### Criterio de evaluación asociado
> **(CE c)** Se han identificado la estructura, variables y relaciones de los conjuntos de datos aplicando técnicas exploratorias.

In [ ]:
# Librerías necesarias para esta sesión
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Configuración de visualización
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Datasets
titanic  = sns.load_dataset('titanic')
penguins = sns.load_dataset('penguins').dropna()
tips     = sns.load_dataset('tips')

print('✅ Librerías y datasets cargados correctamente')

---
## 2.6 Herramientas de visualización: matplotlib y seaborn

En Python existe un ecosistema de librerías para visualización. Las dos más importantes en análisis de datos son:

| Librería | Nivel | Fortaleza | Cuándo usarla |
|----------|-------|-----------|---------------|
| **matplotlib** | Bajo nivel | Control total sobre cada elemento del gráfico | Gráficos personalizados, dashboards, combinar múltiples subplots |
| **seaborn** | Alto nivel (sobre matplotlib) | Gráficos estadísticos con muy poco código | AED, visualizaciones estándar con estética cuidada |

> 💡 En la práctica, **seaborn para el análisis exploratorio** y **matplotlib para ajustar detalles** (títulos, ejes, leyendas, colores exactos). Las dos son complementarias.

### Anatomía de un gráfico en matplotlib

```
Figure  →  el lienzo completo (puede contener varios Axes)
  └── Axes  →  un único gráfico dentro del lienzo
        ├── Title   →  título del gráfico
        ├── X/YLabel →  etiquetas de los ejes
        ├── X/YTick  →  marcas y etiquetas de los ejes
        ├── Legend  →  leyenda
        └── Artist  →  líneas, barras, puntos, texto...
```

### 📌 Ejemplo 1 — matplotlib: Figure y Axes

In [ ]:
# Forma recomendada: crear Figure y Axes explícitamente
fig, ax = plt.subplots(figsize=(9, 4))

# Datos
edad_media_por_clase = titanic.groupby('pclass')['age'].mean()

# Gráfico
bars = ax.bar(
    x=['1ª clase', '2ª clase', '3ª clase'],
    height=edad_media_por_clase.values,
    color=['#4C72B0', '#55A868', '#C44E52'],
    edgecolor='white',
    linewidth=0.8
)

# Personalización mediante el objeto Axes
ax.set_title('Edad media por clase de pasajero — Titanic', fontsize=13, pad=12)
ax.set_xlabel('Clase', fontsize=11)
ax.set_ylabel('Edad media (años)', fontsize=11)
ax.set_ylim(0, 45)
ax.axhline(y=titanic['age'].mean(), color='gray', linestyle='--', alpha=0.6, label='Media global')
ax.legend()

# Etiquetas de valor encima de cada barra
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.5,
            f'{height:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

### 📌 Ejemplo 2 — matplotlib: múltiples subplots con `gridspec`

In [ ]:
# gridspec permite crear layouts irregulares (un gráfico grande + varios pequeños)
fig = plt.figure(figsize=(14, 6))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

ax_main  = fig.add_subplot(gs[:, 0:2])  # Ocupa las 2 primeras columnas, ambas filas
ax_top   = fig.add_subplot(gs[0, 2])
ax_bot   = fig.add_subplot(gs[1, 2])

# Gráfico principal: scatter age vs fare
scatter_data = titanic.dropna(subset=['age', 'fare'])
colores = scatter_data['survived'].map({0: '#C44E52', 1: '#55A868'})
ax_main.scatter(scatter_data['age'], scatter_data['fare'],
                c=colores, alpha=0.5, s=30, edgecolors='none')
ax_main.set_title('Edad vs Precio del billete', fontsize=12)
ax_main.set_xlabel('Edad (años)')
ax_main.set_ylabel('Precio (£)')
from matplotlib.patches import Patch
leyenda = [Patch(color='#55A868', label='Sobrevivió'),
           Patch(color='#C44E52', label='No sobrevivió')]
ax_main.legend(handles=leyenda)

# Top-right: distribución de edad
ax_top.hist(titanic['age'].dropna(), bins=25, color='steelblue', edgecolor='white')
ax_top.set_title('Distribución de Age', fontsize=10)
ax_top.set_xlabel('Edad')

# Bottom-right: supervivencia por clase
surv_clase = titanic.groupby('pclass')['survived'].mean()
ax_bot.bar(['1ª', '2ª', '3ª'], surv_clase.values,
           color=['#4C72B0', '#55A868', '#C44E52'], edgecolor='white')
ax_bot.set_title('Superv. por clase', fontsize=10)
ax_bot.set_ylim(0, 1)
ax_bot.set_ylabel('Proporción')

fig.suptitle('Dashboard exploratorio — Titanic', fontsize=14, y=1.01)
plt.show()

---
### seaborn: visualización estadística de alto nivel

seaborn organiza sus funciones en tres familias:

| Familia | Función principal | Para qué |
|---------|-------------------|----------|
| **Distribuciones** | `histplot`, `kdeplot`, `ecdfplot`, `rugplot` | Distribución de una variable |
| **Relaciones** | `scatterplot`, `lineplot`, `relplot` | Relación entre dos variables numéricas |
| **Categóricas** | `barplot`, `boxplot`, `violinplot`, `stripplot`, `countplot` | Variable numérica según categorías |
| **Matrices** | `heatmap`, `clustermap` | Matrices de correlación o datos tabulados |
| **Grid** | `pairplot`, `FacetGrid`, `catplot`, `displot` | Múltiples subplots automáticos |

### 📌 Ejemplo 3 — seaborn: temas y paletas de color

In [ ]:
# seaborn tiene 5 temas predefinidos
temas = ['white', 'whitegrid', 'dark', 'darkgrid', 'ticks']

fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)

for ax, tema in zip(axes, temas):
    with sns.axes_style(tema):
        ax_temp = ax
        sns.histplot(data=tips, x='total_bill', bins=15, ax=ax_temp, color='steelblue')
        ax_temp.set_title(f"style='{tema}'", fontsize=9)
        ax_temp.set_xlabel('')

plt.suptitle('Temas disponibles en seaborn', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Paletas de color en seaborn
# Tipos principales:
#   - Cualitativas (categorías sin orden): 'Set1', 'Set2', 'tab10', 'pastel'
#   - Secuenciales (valores de menor a mayor): 'Blues', 'YlOrRd', 'viridis'
#   - Divergentes (valores en torno a un centro): 'coolwarm', 'RdYlGn', 'bwr'

fig, axes = plt.subplots(2, 3, figsize=(14, 5))

paletas = [
    ('Set2',     'Cualitativa — Set2',     titanic['pclass']),
    ('tab10',    'Cualitativa — tab10',    titanic['pclass']),
    ('pastel',   'Cualitativa — pastel',   titanic['pclass']),
    ('Blues',    'Secuencial — Blues',     titanic['pclass']),
    ('YlOrRd',   'Secuencial — YlOrRd',   titanic['pclass']),
    ('coolwarm', 'Divergente — coolwarm',  titanic['pclass']),
]

for ax, (palette, titulo, _) in zip(axes.flat, paletas):
    sns.countplot(data=titanic, x='pclass', hue='pclass', palette=palette, ax=ax, legend=False)
    ax.set_title(titulo, fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('Ejemplos de paletas de color en seaborn', y=1.01)
plt.tight_layout()
plt.show()

### 📌 Ejemplo 4 — seaborn: FacetGrid para comparar grupos

In [ ]:
# FacetGrid permite crear una cuadrícula de gráficos idénticos
# separados por valores de una variable categórica
g = sns.FacetGrid(
    penguins, col='species', hue='sex',
    col_wrap=3, height=3.5, aspect=1.1
)
g.map_dataframe(sns.scatterplot, x='bill_length_mm', y='body_mass_g', alpha=0.7)
g.add_legend(title='Sexo')
g.set_axis_labels('Longitud del pico (mm)', 'Masa corporal (g)')
g.set_titles(col_template='Especie: {col_name}')
g.figure.suptitle('Longitud del pico vs Masa corporal — por especie y sexo', y=1.02)
plt.show()

### 📌 Ejemplo 5 — seaborn: `catplot` y `displot` (figuras de alto nivel)

In [ ]:
# catplot es la función de alto nivel para gráficos categóricos
# El parámetro 'kind' controla el tipo: 'strip', 'swarm', 'box', 'violin', 'bar', 'count'
g = sns.catplot(
    data=tips, kind='box',
    x='day', y='total_bill', hue='sex',
    palette='pastel', height=4, aspect=1.6
)
g.set_axis_labels('Día de la semana', 'Total de la cuenta ($)')
g.legend.set_title('Sexo')
g.figure.suptitle('Distribución del total de la cuenta por día y sexo', y=1.02)
plt.show()

In [ ]:
# displot es la función de alto nivel para distribuciones
# kind: 'hist', 'kde', 'ecdf'
g = sns.displot(
    data=penguins, x='flipper_length_mm',
    hue='species', kind='kde',
    fill=True, alpha=0.4, height=4, aspect=2
)
g.set_axis_labels('Longitud de aleta (mm)', 'Densidad')
g.figure.suptitle('Distribución de longitud de aleta por especie (KDE)', y=1.02)
plt.show()

### 📌 Ejemplo 6 — Combinar seaborn y matplotlib: ajuste fino

In [ ]:
# seaborn crea el gráfico, matplotlib ajusta los detalles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: boxplot con anotación personalizada
sns.boxplot(data=titanic, x='pclass', y='age', palette='Set2',
            hue='pclass', legend=False, ax=axes[0])
axes[0].set_title('Distribución de edad por clase', fontsize=12)
axes[0].set_xlabel('Clase de pasajero')
axes[0].set_ylabel('Edad (años)')
axes[0].set_xticklabels(['1ª clase', '2ª clase', '3ª clase'])

# Añadimos la mediana como texto con matplotlib
medianas = titanic.groupby('pclass')['age'].median()
for i, (clase, mediana) in enumerate(medianas.items()):
    axes[0].text(i, mediana + 1, f'Md={mediana:.0f}',
                 ha='center', va='bottom', fontsize=9, color='black')

# Panel derecho: heatmap con máscara
num_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr = titanic[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))   # Solo triángulo inferior
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=axes[1])
axes[1].set_title('Correlación entre variables numéricas', fontsize=12)
# Rotamos las etiquetas para que no se solapen
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

---
### 🔧 Ejercicio 1 — Dashboard visual de un dataset

Crea un dashboard de 4 gráficos en una sola Figure para el dataset `tips` de seaborn.

In [ ]:
# El dataset 'tips' contiene propinas en un restaurante
# tips = sns.load_dataset('tips')
# Columnas: total_bill, tip, sex, smoker, day, time, size

# Crea una Figure con 2x2 subplots que incluya:
#   - Top-left    : histograma de 'total_bill' con KDE
#   - Top-right   : boxplot de 'tip' agrupado por 'day'
#   - Bottom-left : scatter de 'total_bill' vs 'tip', coloreado por 'sex'
#   - Bottom-right: countplot de 'day' coloreado por 'time'

# fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# ...
# plt.suptitle('Dashboard — Tips dataset', fontsize=14)
# plt.tight_layout()
# plt.show()

print('Completa la figura con los 4 subplots indicados.')

---
## 2.7 Perfilado automático de datos: ydata-profiling

Hacer el AED manualmente tarda tiempo. La librería **ydata-profiling** (antes `pandas-profiling`) genera un informe HTML completo con un solo comando, cubriendo:

- Descripción general del dataset (filas, columnas, tipos, duplicados, memoria)
- Análisis por variable: distribución, estadísticos, valores ausentes, alertas
- Análisis de correlaciones (Pearson, Spearman, Cramér)
- Análisis de interacciones entre variables
- Muestra de datos

> ⚠️ **ydata-profiling no sustituye al AED manual**: es un punto de partida rápido para detectar problemas. El análisis profundo sigue requiriendo código y criterio.

In [ ]:
# Instalamos ydata-profiling si no está disponible (ejecutar solo una vez)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'ydata-profiling', '-q'],
    capture_output=True, text=True
)
print('✅ ydata-profiling instalado.' if result.returncode == 0
      else f'❌ Error: {result.stderr[:200]}')

### 📌 Ejemplo 7 — Generar un informe de perfilado básico

In [ ]:
from ydata_profiling import ProfileReport
# Generamos el informe sobre el dataset Titanic
# minimal=False → informe completo (más lento en datasets grandes)
profile = ProfileReport(
    titanic,
    title='Informe AED — Titanic',
    explorative=True
)

print('Informe generado. Guardando como HTML...')

# Guardamos el informe como archivo HTML
profile.to_file('informe_titanic.html')
print('✅ Informe guardado en: informe_titanic.html')
print('   Ábrelo en tu navegador para explorarlo.')

### 📌 Ejemplo 8 — Visualizar el informe directamente en el notebook

In [ ]:
# También podemos mostrar el informe embebido en el notebook
# (puede tardar unos segundos en renderizarse)
profile.to_notebook_iframe()

### 📌 Ejemplo 9 — Opciones del informe: minimal y explorative

In [ ]:
# Para datasets grandes, usar minimal=True es mucho más rápido
# (omite correlaciones y algunas visualizaciones costosas)

# Ejemplo con el dataset penguins + modo minimal
profile_penguins = ProfileReport(
    sns.load_dataset('penguins'),
    title='Informe AED — Penguins (minimal)',
    minimal=True
)
profile_penguins.to_file('informe_penguins.html')
print('✅ informe_penguins.html guardado.')

# Resumen de las alertas detectadas automáticamente
print('\n🔔 Alertas detectadas:')
for alerta in profile_penguins.get_description().alerts:
    print(f'  · {alerta}')

### 📌 Ejemplo 10 — Comparar dos versiones de un dataset

In [ ]:
# ydata-profiling permite comparar dos DataFrames (antes/después de limpiar)
# Simulamos un dataset «antes» y otro «después» de eliminar outliers

titanic_antes  = titanic.copy()

# «Después»: eliminamos filas con fare > 300 y age > 70
titanic_despues = titanic[
    (titanic['fare'] <= 300) & (titanic['age'] <= 70)
].copy()

print(f'Dataset antes  : {titanic_antes.shape}')
print(f'Dataset después: {titanic_despues.shape}')

profile_antes   = ProfileReport(titanic_antes,   title='Antes (con outliers)',   minimal=True)
profile_despues = ProfileReport(titanic_despues, title='Después (sin outliers)', minimal=True)

# Informe comparativo
comparison_report = profile_antes.compare(profile_despues)
comparison_report.to_file('informe_comparativo.html')
print('\n✅ Informe comparativo guardado en: informe_comparativo.html')

### 📌 Ejemplo 11 — Extraer métricas del informe programáticamente

In [ ]:
# get_description() devuelve un objeto con todas las métricas del informe
desc = profile.get_description()

# Estadísticos globales del dataset
stats = desc.table
print('Estadísticos globales del dataset Titanic:')
print(f'  Nº de variables   : {stats["n_var"]}')
print(f'  Nº de observaciones: {stats["n"]}')
print(f'  Celdas totales    : {stats["n_cells_missing"] + stats["n"] * stats["n_var"]}')
print(f'  Celdas ausentes   : {stats["n_cells_missing"]} ({stats["p_cells_missing"]*100:.1f}%)')
print(f'  Filas duplicadas  : {stats["n_duplicates"]}')

# Estadísticos por variable
print('\nResumen por variable (primeras 5):')
vars_desc = desc.variables
for nombre, var_stats in list(vars_desc.items())[:5]:
    tipo = var_stats.get('type', '?')
    n_missing = var_stats.get('n_missing', 0)
    print(f'  {nombre:15s} → tipo: {str(tipo):20s} ausentes: {n_missing}')

---
### 🔧 Ejercicio 2 — Perfilado automático de un dataset propio

Genera un informe de perfilado completo para el dataset de tu elección y extrae conclusiones.

In [ ]:
# TAREA 1: Elige un dataset (tips, diamonds, penguins, o uno externo)
# df_ej2 = sns.load_dataset('tips')

# TAREA 2: Genera el informe de perfilado completo
# profile_ej2 = ProfileReport(df_ej2, title='Mi informe', explorative=True)

# TAREA 3: Guárdalo como HTML y ábrelo en el navegador
# profile_ej2.to_file('mi_informe.html')

# TAREA 4: Lista las alertas automáticas que detecta
# for alerta in profile_ej2.get_description().alerts:
#     print(alerta)

# TAREA 5 (reflexión): Basándote en el informe, responde:
#   a) ¿Qué variables tienen valores ausentes? ¿En qué porcentaje?
#   b) ¿Hay variables con alta correlación entre sí?
#   c) ¿Alguna variable tiene distribución muy asimétrica?
#   d) ¿Detecta el informe outliers o variables constantes?
#   Respuesta: ...

print('Completa las tareas comentadas arriba.')

---
### 🔧 Ejercicio 3 — AED visual completo: del gráfico a la conclusión

A partir del dataset `tips`, crea los gráficos necesarios para responder a las siguientes preguntas. Para cada gráfico, incluye debajo una celda Markdown con tu interpretación.

In [ ]:
# PREGUNTA 1: ¿Los clientes que fuman dejan más propina?
#   → Gráfico sugerido: violinplot o boxplot de 'tip' agrupado por 'smoker'

# PREGUNTA 2: ¿El tamaño del grupo (size) influye en el total de la cuenta?
#   → Gráfico sugerido: scatter de 'size' vs 'total_bill' o boxplot agrupado

# PREGUNTA 3: ¿Qué día de la semana y turno genera más ingresos al restaurante?
#   → Gráfico sugerido: barplot de total_bill agrupado por day y hue=time

# PREGUNTA 4: ¿Cuál es la relación entre el total de la cuenta y la propina?
#   → Gráfico sugerido: scatter de 'total_bill' vs 'tip' con regresión (sns.lmplot)

# Para cada pregunta:
#   1. Dibuja el gráfico con matplotlib/seaborn
#   2. Responde la pregunta en una celda Markdown bajo el gráfico

print('Completa los gráficos y las interpretaciones.')

---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| matplotlib | Bajo nivel. Control total: Figure, Axes, Artists. Imprescindible para layouts personalizados. |
| seaborn | Alto nivel. Gráficos estadísticos con poco código. Funciones de alto nivel: `catplot`, `displot`, `pairplot`. |
| Temas seaborn | `sns.set_theme(style=...)`: white, whitegrid, dark, darkgrid, ticks. |
| Paletas de color | Cualitativas (categorías), secuenciales (gradiente), divergentes (centro). Elegir según el tipo de dato. |
| `FacetGrid` | Cuadrícula de gráficos iguales separados por categoría. |
| ydata-profiling | `ProfileReport(df).to_file('informe.html')` → informe AED completo en una línea. |
| Alertas automáticas | `profile.get_description().alerts` → lista de problemas detectados (ausentes, alta cardinalidad, outliers...). |
| Informes comparativos | `profile1.compare(profile2)` → comparar antes/después de limpiar. |

### 🔜 Próximo bloque
**Bloque 3 — Sesión 1:** Detección de problemas de calidad: valores ausentes (tipos MCAR, MAR, MNAR) y detección con `isnull` y `missingno`.